# 🏥 BÁO CÁO CHẨN ĐOÁN DỮ LIỆU (DATA DIAGNOSTIC REPORT)
**Mục tiêu:** Quét toàn diện file `final_master_data.csv` để phát hiện các vấn đề về định dạng, khuyết thiếu, ngoại lệ và vi phạm quy tắc kinh doanh.
**Lưu ý:** File này CHỈ BÁO CÁO (đếm số lượng, tính tỷ lệ), TUYỆT ĐỐI KHÔNG XÓA hay THAY ĐỔI dữ liệu gốc.

In [1]:
import pandas as pd
import numpy as np

# Đọc dữ liệu
data_dir = '../data/'
df = pd.read_csv(f'{data_dir}final_master_data.csv', low_memory=False)

print(f"Tổng quan dữ liệu: {df.shape[0]} dòng và {df.shape[1]} cột")

# Ép kiểu ngày tháng để phục vụ kiểm tra logic thời gian
date_cols = ['order_date', 'ship_date', 'delivery_date', 'return_date', 'review_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

Tổng quan dữ liệu: 714673 dòng và 86 cột


## 1. Kiểm tra và Chuẩn hóa Chuỗi văn bản (Text Standardization)
Đưa tất cả các cột dạng chuỗi (category, city, status...) về chữ thường và xóa khoảng trắng dư thừa để tránh việc Pandas nhận diện sai (VD: "Hà Nội" và "hà nội " bị coi là 2 thành phố khác nhau).

In [2]:
# Tìm tất cả các cột có kiểu dữ liệu là chuỗi (object)
text_cols = df.select_dtypes(include=['object']).columns

# Chuẩn hóa
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()
    # Chuyển lại chữ 'nan' thành np.nan để đếm Missing Values chính xác
    df[col] = df[col].replace('nan', np.nan)

print("✅ Đã chuẩn hóa xong văn bản (Chữ thường, xóa khoảng trắng)!")

✅ Đã chuẩn hóa xong văn bản (Chữ thường, xóa khoảng trắng)!


## 2. Báo cáo Dữ liệu Khuyết thiếu (Missing Values)
Thống kê các cột bị trống dữ liệu. Dựa vào bảng này, nhóm sẽ quyết định chiến lược điền (Fill/Impute) hoặc xóa (Drop).

In [3]:
# Tính toán số lượng và tỷ lệ Missing
missing_data = df.isnull().sum().reset_index()
missing_data.columns = ['Tên Cột', 'Số dòng thiếu']
missing_data['Tỷ lệ thiếu (%)'] = (missing_data['Số dòng thiếu'] / len(df)) * 100

# Lọc ra các cột có dữ liệu thiếu và sắp xếp
missing_report = missing_data[missing_data['Số dòng thiếu'] > 0].sort_values(by='Tỷ lệ thiếu (%)', ascending=False)

print("--- 🚨 CÁC CỘT BỊ THIẾU DỮ LIỆU ---")
display(missing_report.style.format({'Tỷ lệ thiếu (%)': '{:.2f}%'}).background_gradient(cmap='Reds'))

--- 🚨 CÁC CỘT BỊ THIẾU DỮ LIỆU ---


,Tên Cột,Số dòng thiếu,Tỷ lệ thiếu (%)
60,promo2_promo_name,714467,99.97%
65,promo2_applicable_category,714467,99.97%
61,promo2_promo_type,714467,99.97%
62,promo2_discount_value,714467,99.97%
63,promo2_start_date,714467,99.97%
64,promo2_end_date,714467,99.97%
6,promo_id_2,714467,99.97%
66,promo2_promo_channel,714467,99.97%
67,promo2_stackable_flag,714467,99.97%
68,promo2_min_order_value,714467,99.97%


In [4]:
import pandas as pd
import numpy as np

# 1. Đọc dữ liệu
data_dir = '../data/'
df = pd.read_csv(f'{data_dir}final_master_data.csv', low_memory=False)

print("--- BẮT ĐẦU XỬ LÝ MISSING VALUES ---")

# ==========================================
# 1. NHÓM KHUYẾN MÃI (PROMO 1 & PROMO 2)
# Khách không dùng mã -> Tiền giảm = 0, Text = 'No Promo'
# ==========================================
promo_text_cols = [
    'promo1_promo_name', 'promo1_promo_type', 'promo1_applicable_category', 
    'promo1_promo_channel', 'promo1_start_date', 'promo1_end_date', 'promo_id', 
    'promo_id_2', 'promo2_promo_name', 'promo2_promo_type', 
    'promo2_applicable_category', 'promo2_promo_channel', 'promo2_start_date', 'promo2_end_date'
]
promo_num_cols = [
    'promo1_discount_value', 'promo1_stackable_flag', 'promo1_min_order_value', 
    'promo2_discount_value', 'promo2_stackable_flag', 'promo2_min_order_value'
]

for col in promo_text_cols:
    if col in df.columns: df[col] = df[col].fillna('No Promo')
        
for col in promo_num_cols:
    if col in df.columns: df[col] = df[col].fillna(0)

# ==========================================
# 2. NHÓM TRẢ HÀNG (RETURNS)
# Khách không trả hàng -> Số lượng = 0, Lý do = 'Not Returned'
# ==========================================
if 'return_id' in df.columns: df['return_id'] = df['return_id'].fillna('Not Returned')
if 'return_reason' in df.columns: df['return_reason'] = df['return_reason'].fillna('Not Returned')
if 'return_quantity' in df.columns: df['return_quantity'] = df['return_quantity'].fillna(0)
if 'refund_amount' in df.columns: df['refund_amount'] = df['refund_amount'].fillna(0)
# Cột return_date để trống (NaT) cũng được, hoặc điền chuỗi
if 'return_date' in df.columns: df['return_date'] = df['return_date'].fillna('1900-01-01') 

# ==========================================
# 3. NHÓM ĐÁNH GIÁ (REVIEWS)
# Khách không đánh giá -> Rating = 0, Text = 'No Review'
# ==========================================
if 'review_id' in df.columns: df['review_id'] = df['review_id'].fillna('No Review')
if 'review_title' in df.columns: df['review_title'] = df['review_title'].fillna('No Review')
if 'reviewer_customer_id' in df.columns: df['reviewer_customer_id'] = df['reviewer_customer_id'].fillna('No Reviewer')
if 'rating' in df.columns: df['rating'] = df['rating'].fillna(0)
if 'review_date' in df.columns: df['review_date'] = df['review_date'].fillna('1900-01-01')

# ==========================================
# 4. NHÓM VẬN CHUYỂN
# ==========================================

    
# Lưu ý: ship_date và delivery_date bị thiếu (12.49%) thường là đơn bị hủy (Cancelled).
# Tạm thời điền ngày mặc định để không bị lỗi lúc tính toán.
if 'ship_date' in df.columns: df['ship_date'] = df['ship_date'].fillna('1900-01-01')
if 'delivery_date' in df.columns: df['delivery_date'] = df['delivery_date'].fillna('1900-01-01')

# ==========================================
# 5. NHÓM VẬN HÀNH (WEB TRAFFIC & INVENTORY)
# Tỷ lệ thiếu rất nhỏ (< 5%), ta dùng Trung vị (Median) để điền
# ==========================================
op_cols = [
    'sessions', 'unique_visitors', 'page_views', 'bounce_rate', 'avg_session_duration_sec', 
    'stock_on_hand', 'units_received', 'units_sold', 'stockout_days', 'days_of_supply', 
    'fill_rate', 'stockout_flag', 'overstock_flag', 'reorder_flag', 'sell_through_rate'
]
for col in op_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# ==========================================
# KIỂM TRA LẠI THÀNH QUẢ
# ==========================================
print("✅ HOÀN TẤT ĐIỀN MISSING VALUES!")
missing_left = df.isnull().sum()
missing_left = missing_left[missing_left > 0]

if len(missing_left) == 0:
    print("🎉 Tuyệt vời! File dữ liệu hiện tại không còn bất kỳ ô trống (NaN) !")
else:
    print("⚠️ Vẫn còn sót các cột sau bị trống:")
    print(missing_left)

# Lưu đè lại vào biến df để chuẩn bị làm Outlier
# Chưa xuất file vội vì mình còn làm Outlier nữa

--- BẮT ĐẦU XỬ LÝ MISSING VALUES ---
✅ HOÀN TẤT ĐIỀN MISSING VALUES!
⚠️ Vẫn còn sót các cột sau bị trống:
shipping_fee    89287
dtype: int64


## 3. Báo cáo Dữ liệu Ngoại lệ (Outliers - Phương pháp IQR)
Quét các cột dạng số (Numeric) để tìm ra các giá trị lớn/nhỏ bất thường.
*Lưu ý: Outlier có thể là lỗi nhập liệu, hoặc có thể là đơn hàng B2B siêu lớn. Cần xem xét kỹ trước khi xóa.*

In [5]:
import numpy as np

print("\n--- BƯỚC 3: XỬ LÝ NGOẠI LỆ (OUTLIERS) ---")

# 3.1. Nhóm an toàn để Cắt ngọn (Capping) - Dùng np.clip
safe_to_cap = ['avg_session_duration_sec']

for col in safe_to_cap:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = Q3 + 1.5 * IQR
        
        # Đếm trước khi cắt
        outlier_count = len(df[df[col] > upper_bound])
        
        # Cắt ngọn trần (những số lớn hơn upper_bound sẽ bị kéo về bằng upper_bound)
        df[col] = np.clip(df[col], a_min=None, a_max=upper_bound)
        print(f"✅ Đã cắt ngọn {outlier_count} dòng quá khổ cho cột: {col}")

# 3.2. Nhóm Số lượng (Quantity) -> Không cắt, tạo cờ (Flag) "Đơn hàng sỉ"
if 'quantity' in df.columns:
    Q1_qty = df['quantity'].quantile(0.25)
    Q3_qty = df['quantity'].quantile(0.75)
    IQR_qty = Q3_qty - Q1_qty
    upper_bound_qty = Q3_qty + 1.5 * IQR_qty
    
    # Tạo thêm 1 cột mới: 1 là đơn sỉ, 0 là đơn lẻ bình thường
    df['is_bulk_order'] = np.where(df['quantity'] > upper_bound_qty, 1, 0)
    bulk_count = df['is_bulk_order'].sum()
    print(f"✅ Đã giữ nguyên số lượng và tạo cờ 'is_bulk_order' cho {bulk_count} đơn hàng mua sỉ.")

print("⚠️ Đã bỏ qua cắt ngọn với 'unit_price', 'cogs' để bảo toàn dữ liệu tài chính.")


--- BƯỚC 3: XỬ LÝ NGOẠI LỆ (OUTLIERS) ---
✅ Đã cắt ngọn 0 dòng quá khổ cho cột: avg_session_duration_sec
✅ Đã giữ nguyên số lượng và tạo cờ 'is_bulk_order' cho 0 đơn hàng mua sỉ.
⚠️ Đã bỏ qua cắt ngọn với 'unit_price', 'cogs' để bảo toàn dữ liệu tài chính.


## 4. Kiểm định Quy tắc Kinh doanh (Business Logic Validation)
Đếm số lượng các dòng vi phạm logic cốt lõi. Đây là các "Dữ liệu Rác" (Anomalies) chắc chắn phải xử lý hoặc loại bỏ ở bước sau.

In [6]:
print("--- 🕵️‍♂️ BÁO CÁO VI PHẠM LOGIC KINH DOANH ---")

# 1. Giá vốn >= Giá bán (Bán lỗ vốn sai quy định)
if 'cogs' in df.columns and 'unit_price' in df.columns:
    err_cogs = len(df[df['cogs'] >= df['unit_price']])
    print(f"❌ [cogs >= price]: Có {err_cogs} dòng.")

# 2. Tiền giảm giá > Tiền hàng
if 'discount_amount' in df.columns and 'quantity' in df.columns and 'unit_price' in df.columns:
    err_discount = len(df[df['discount_amount'] > (df['quantity'] * df['unit_price'])])
    print(f"❌ [discount > total_value]: Có {err_discount} dòng.")

# 3. Trả hàng nhiều hơn mua
if 'return_quantity' in df.columns and 'quantity' in df.columns:
    err_return = len(df[df['return_quantity'] > df['quantity']])
    print(f"❌ [return_qty > qty]: Có {err_return} dòng.")

# 4. Thời gian xuyên không (Ship trước khi đặt)
if 'ship_date' in df.columns and 'order_date' in df.columns:
    err_ship = len(df[df['ship_date'] < df['order_date']])
    print(f"❌ [ship_date < order_date]: Có {err_ship} dòng.")

# 5. Điểm đánh giá (Rating) ngoài khoảng 1-5
if 'rating' in df.columns:
    # Lọc bỏ các dòng NaN trước khi check
    valid_ratings = df['rating'].dropna()
    err_rating = len(valid_ratings[~valid_ratings.isin([1.0, 2.0, 3.0, 4.0, 5.0])])
    print(f"❌ [rating không thuộc 1-5]: Có {err_rating} dòng.")

# 6. Trả góp (Installments) nhỏ hơn 1
if 'installments' in df.columns:
    err_installments = len(df[df['installments'] < 1])
    print(f"❌ [installments < 1]: Có {err_installments} dòng.")

# 7. Phí ship âm
if 'shipping_fee' in df.columns:
    err_shipping_fee = len(df[df['shipping_fee'] < 0])
    print(f"❌ [shipping_fee < 0]: Có {err_shipping_fee} dòng.")

--- 🕵️‍♂️ BÁO CÁO VI PHẠM LOGIC KINH DOANH ---
❌ [cogs >= price]: Có 133052 dòng.
❌ [discount > total_value]: Có 0 dòng.
❌ [return_qty > qty]: Có 1 dòng.
❌ [ship_date < order_date]: Có 89287 dòng.
❌ [rating không thuộc 1-5]: Có 601120 dòng.
❌ [installments < 1]: Có 0 dòng.
❌ [shipping_fee < 0]: Có 0 dòng.


In [7]:
print("\n--- BƯỚC 4: LỌC RÁC PHI LOGIC KINH DOANH ---")
initial_rows = len(df)

# 1. Bán lỗ vốn sai quy định (Giá vốn >= Giá bán)
if 'cogs' in df.columns and 'unit_price' in df.columns:
    df = df[df['cogs'] < df['unit_price']]

# 2. Tiền giảm giá > Tiền hàng
if 'discount_amount' in df.columns and 'quantity' in df.columns and 'unit_price' in df.columns:
    df = df[df['discount_amount'] <= (df['quantity'] * df['unit_price'])]

# 3. Trả hàng nhiều hơn mua
if 'return_quantity' in df.columns and 'quantity' in df.columns:
    df = df[df['return_quantity'] <= df['quantity']]

# 4. Giao hàng xuyên không (Ship trước khi đặt)
if 'ship_date' in df.columns and 'order_date' in df.columns:
    # Lọc những đơn ship chuẩn HOẶC những đơn bị hủy (mình đã điền 1900-01-01 ở bước Missing Value)
    df = df[(df['ship_date'] >= df['order_date']) | (df['ship_date'] == pd.to_datetime('1900-01-01'))]

# 5. Điểm đánh giá (Rating) ngoài khoảng 1-5 (Bao gồm số 0 mình tự điền)
if 'rating' in df.columns:
    df = df[df['rating'].isin([0.0, 1.0, 2.0, 3.0, 4.0, 5.0])]

final_rows = len(df)
print(f"✅ Đã dọn dẹp logic! Số dòng rác bị xóa vĩnh viễn: {initial_rows - final_rows} dòng.")


# ==========================================
# BƯỚC CUỐI CÙNG: XUẤT KHO LỚP VÀNG (GOLD LAYER)
# ==========================================



--- BƯỚC 4: LỌC RÁC PHI LOGIC KINH DOANH ---
✅ Đã dọn dẹp logic! Số dòng rác bị xóa vĩnh viễn: 205385 dòng.


## 5. Quét và Xóa Dữ Liệu Trùng Lặp (Duplicate Rows)
**Mục tiêu:** Loại bỏ các dòng bị nhân bản (clone) do lỗi hệ thống hoặc do quá trình JOIN bảng gây ra. 
**Quy tắc:** Một đơn hàng (`order_id`) không thể chứa 2 dòng giống hệt nhau cho cùng một sản phẩm (`product_id`). Nếu phát hiện trùng, ta chỉ giữ lại dòng đầu tiên và xóa các dòng bản sao để tránh làm khống doanh thu.

In [8]:
print("\n--- BƯỚC 5: KIỂM TRA VÀ XÓA DÒNG TRÙNG LẶP (DUPLICATES) ---")

# 1. Quét số lượng trùng lặp
total_dups = df.duplicated().sum()
print(f"🔍 Quét toàn hệ thống: Phát hiện {total_dups} dòng trùng lặp 100% tất cả các cột.")

subset_dups = df.duplicated(subset=['order_id', 'product_id']).sum()
print(f"🔍 Quét theo Khóa (Order + Product): Phát hiện {subset_dups} dòng trùng lặp.")

# 2. Xóa dòng trùng lặp (Dựa trên Order ID và Product ID)
initial_rows = len(df)
# keep='first' nghĩa là giữ lại dòng đầu tiên, xóa các dòng copy
df = df.drop_duplicates(subset=['order_id', 'product_id'], keep='first')
final_rows = len(df)

if initial_rows > final_rows:
    print(f"✅ Đã dọn dẹp sạch sẽ! Xóa thành công {initial_rows - final_rows} dòng bản sao.")
else:
    print("✅ Dữ liệu hoàn hảo, không có bất kỳ dòng copy nào!")


# ==========================================
# BƯỚC CUỐI CÙNG: CHỐT SỔ & XUẤT KHO LỚP VÀNG (GOLD LAYER)
# ==========================================
# Lưu ý: Nhớ xóa hoặc comment lại lệnh to_csv ở Bước 4 cũ để tránh xuất file 2 lần nhé!
output_path = f'{data_dir}master_data_preprocessed.csv'
df.to_csv(output_path, index=False)

print(f"\n📦 XUẤT SẮC! File dữ liệu SIÊU SẠCH đã chính thức chốt sổ và lưu tại: {output_path}")


--- BƯỚC 5: KIỂM TRA VÀ XÓA DÒNG TRÙNG LẶP (DUPLICATES) ---
🔍 Quét toàn hệ thống: Phát hiện 0 dòng trùng lặp 100% tất cả các cột.
🔍 Quét theo Khóa (Order + Product): Phát hiện 16 dòng trùng lặp.
✅ Đã dọn dẹp sạch sẽ! Xóa thành công 16 dòng bản sao.

📦 XUẤT SẮC! File dữ liệu SIÊU SẠCH đã chính thức chốt sổ và lưu tại: ../data/master_data_preprocessed.csv
